In [41]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from difflib import get_close_matches

In [42]:
df=pd.read_csv("/content/MusicRecommend.csv")

Data Preprocessing

In [43]:
df.head()

,Song-Name,Singer/Artists,Genre,Album/Movie,User-Rating
0,Aankh Marey,"Kumar Sanu, Mika Singh, Neha Kakkar",BollywoodDance,Simmba,8.8/10
1,Coca Cola,"Neha Kakkar, Tony Kakkar",BollywoodDanceRomantic,Luka Chuppi,9.0/10
2,Apna Time Aayega,Ranveer Singh,BollywoodDance,Gully Boy,9.7/10
3,Mungda,"Jyotica Tangri, Shaan, Subhro Ganguly",BollywoodDance,Total Dhamaal,9.1/10
4,Tere Bin,"Asees Kaur, Rahat Fateh Ali Khan, Tanishk Bagchi",BollywoodRomantic,Simmba,9.2/10


In [44]:
df.shape

(2420, 5)

In [45]:
df.isnull().sum()

,0
Song-Name,0
Singer/Artists,10
Genre,0
Album/Movie,3
User-Rating,0


In [46]:
df.dropna(inplace=True)

In [47]:
df.isnull().sum()

,0
Song-Name,0
Singer/Artists,0
Genre,0
Album/Movie,0
User-Rating,0


In [48]:
df.duplicated().sum()

np.int64(16)

In [49]:
df = df.drop_duplicates()

In [50]:
df.columns

Index(['Song-Name', 'Singer/Artists', 'Genre', 'Album/Movie', 'User-Rating'], dtype='object')

In [51]:
df.dtypes

,0
Song-Name,object
Singer/Artists,object
Genre,object
Album/Movie,object
User-Rating,object


In [52]:
df['User-Rating'] = df['User-Rating'].str.replace('/10', '', regex=False).astype(float)

In [53]:
df["User-Rating"].head()

,User-Rating
0,8.8
1,9.0
2,9.7
3,9.1
4,9.2


In [54]:
df['Genre'] = df['Genre'].apply(lambda x: re.sub(r'([a-z])([A-Z])', r'\1 \2', str(x)))

In [55]:
df['Genre'].head()

,Genre
0,Bollywood Dance
1,Bollywood Dance Romantic
2,Bollywood Dance
3,Bollywood Dance
4,Bollywood Romantic


In [56]:
df['Singer/Artists'] = df['Singer/Artists'].str.replace(' ', '')

In [57]:
df['Singer/Artists'] = df['Singer/Artists'].str.replace(',', ' ')

In [58]:
df['Singer/Artists'].head()

,Singer/Artists
0,KumarSanu MikaSingh NehaKakkar
1,NehaKakkar TonyKakkar
2,RanveerSingh
3,JyoticaTangri Shaan SubhroGanguly
4,AseesKaur RahatFatehAliKhan TanishkBagchi


In [59]:
df['Singer/Artists'] = df['Singer/Artists'].str.lower()
df['Genre'] = df['Genre'].str.lower()
df['Album/Movie'] = df['Album/Movie'].str.lower()

In [60]:
df.head()

,Song-Name,Singer/Artists,Genre,Album/Movie,User-Rating
0,Aankh Marey,kumarsanu mikasingh nehakakkar,bollywood dance,simmba,8.8
1,Coca Cola,nehakakkar tonykakkar,bollywood dance romantic,luka chuppi,9.0
2,Apna Time Aayega,ranveersingh,bollywood dance,gully boy,9.7
3,Mungda,jyoticatangri shaan subhroganguly,bollywood dance,total dhamaal,9.1
4,Tere Bin,aseeskaur rahatfatehalikhan tanishkbagchi,bollywood romantic,simmba,9.2


In [61]:
df['Song-Name'] = df['Song-Name'].str.strip().str.lower()

In [62]:
df.dtypes

,0
Song-Name,object
Singer/Artists,object
Genre,object
Album/Movie,object
User-Rating,float64


In [63]:
df = df.reset_index(drop=True)

In [64]:
#different weights
df['combined'] = (
    df['Genre'] + " " + df['Genre'] + " " + df['Genre'] + " " +
    df['Singer/Artists'] + " " + df['Singer/Artists'] + " " +
    df['Album/Movie']
).str.strip()

Vectorization, Cosine Similarity

In [65]:
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
matrix = vectorizer.fit_transform(df['combined'])

In [66]:
similarity = cosine_similarity(matrix)

In [67]:
scaler = MinMaxScaler()
df['Rating-Normalized'] = scaler.fit_transform(df[['User-Rating']])

Recommendation Function

In [68]:
def recommend(song_name, top_n=5, sim_weight=0.7, rating_weight=0.3):

    song_name = song_name.strip().lower()

    if song_name not in df['Song-Name'].values:
        close = get_close_matches(song_name, df['Song-Name'].values, n=1, cutoff=0.6)
        if close:
            print(f"⚠️  '{song_name}' not found. Did you mean: '{close[0]}'? Showing results for that.")
            song_name = close[0]
        else:
            print("❌ Song not found. Please check the name and try again.")
            return None

    idx = df[df['Song-Name'] == song_name].index[0]

    sim_scores = similarity[idx]

    hybrid_scores = (sim_weight * sim_scores) + (rating_weight * df['Rating-Normalized'].values)

    scored = pd.DataFrame({
        'index':      np.arange(len(df)),
        'sim_score':  sim_scores,
        'hybrid':     hybrid_scores,
        'Song-Name':  df['Song-Name'].values,
        'Singer/Artists': df['Singer/Artists'].values,
        'Genre':      df['Genre'].values,
        'User-Rating': df['User-Rating'].values,
    })

    scored = scored[scored['index'] != idx]
    scored = scored.sort_values('hybrid', ascending=False)
    top = scored.head(top_n).reset_index(drop=True)

    return top[['Song-Name', 'Singer/Artists', 'Genre', 'User-Rating']]


Evaluation

In [69]:
def coverage(k=5):
    recommended = set()

    for i in range(len(df)):
        scores = list(enumerate(similarity[i]))
        scores = sorted(scores, key=lambda x: x[1], reverse=True)

        top_k = scores[1:k+1]
        for j in top_k:
            recommended.add(j[0])

    return len(recommended) / len(df)

In [70]:
def intra_list_diversity(results):
    if results is None or results.empty:
        return 0.0
    genres = results['Genre'].values
    return len(set(genres)) / len(genres)

In [71]:
def display_recommendations(results, input_song):
    """
    Pretty-prints the recommendation DataFrame.
    [CHANGE] Separated display logic from recommendation logic (single responsibility).
    """
    if results is None or results.empty:
        return
    print(f"\n🎵 Top recommendations for '{input_song}':\n")
    print(f"{'#':<4} {'Song':<35} {'Artist':<25} {'Genre':<20} {'⭐ Rating'}")
    print("─" * 95)
    for rank, (_, row) in enumerate(results.iterrows(), 1):
        print(f"{rank:<4} {row['Song-Name']:<35} {row['Singer/Artists']:<25} {row['Genre']:<20} {row['User-Rating']}/10")
    print()

Testing

In [72]:
coverage()

0.8469259723964868

In [73]:
intra_list_diversity(recommend('coca cola'))

0.8

In [74]:
if __name__ == "__main__":
    # [CHANGE] Wrapped in __name__ guard so imports don't auto-run the loop

    print("\n🎧 Music Recommendation System")
    print("Type a song name to get recommendations, or 'exit' to quit.\n")

    while True:
        user_input = input("🎤 Enter a song: ").strip()
        if user_input.lower() == "exit":
            print("👋 Goodbye!")
            break
        if not user_input:
            continue
        results = recommend(user_input, top_n=5)
        display_recommendations(results, user_input)


🎧 Music Recommendation System
Type a song name to get recommendations, or 'exit' to quit.

🎤 Enter a song: coca cola

🎵 Top recommendations for 'coca cola':

#    Song                                Artist                    Genre                ⭐ Rating
───────────────────────────────────────────────────────────────────────────────────────────────
1    mile ho tum (reprise)               nehakakkar tonykakkar     bollywood romantic   9.0/10
2    helicopter                          nehakakkar tonykakkar     bollywood dance      9.3/10
3    mohabbat nasha hai                  nehakakkar tonykakkar     bollywood sad        9.4/10
4    sawan aaya hai (unplugged version)  nehakakkar tonykakkar     bollywood sad        9.3/10
5    jab se mere naina                   shaan                     bollywood dance romantic 9.6/10

🎤 Enter a song: exit
👋 Goodbye!
